In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path

from dusty_colors.utils import load_jackknives
import matplotlib.patches as mpatches

In [ ]:
def get_color(color):
    return {
        "g-i": "k",
        "g-r": "C0",
        "r-i": "C2",
        "i-z": "C3",
    }[color]

In [ ]:
results = load_jackknives("../results_main", "fcolors")

x = results[f"g-i_bin_centers"]
y_all = -2.5 * np.log10(results[f"g-i_avg"])
y_err_ = 2.5 / np.log(10) * results[f"g-i_err"] / results[f"g-i_avg"]

fig, ax = plt.subplots(figsize=(3, 3), dpi=200)

for i, (xi, yi, ei) in enumerate(zip(x, y_all, y_err_)):
    ls = {0: "-", 1: "--", 2: ":"}[i % 3]
    ax.errorbar(
        xi,
        yi,
        yerr=ei,
        marker=".",
        ls=ls,
        label=f"{i+1}",
        c=f"C{i // 3}",
    )

x_mean = x[0]
y_mean = y_all.mean(axis=0)
y_cov = np.cov(y_all, rowvar=False, bias=True) * (len(y_all) - 1)
y_err = np.sqrt(np.diag(y_cov))
ax.errorbar(
    x_mean,
    y_mean,
    yerr=y_err,
    marker="s",
    markerfacecolor="none",
    markersize=5,
    ls="",
    c="k",
    zorder=10,
)

ax.set(xscale="log", yscale="log")
ax.set(xlabel="$r_\perp$ [Mpc]", ylabel="$E(g-i)$ [mag]")
ax.legend(
    title="Jackknife region",
    ncols=3,
    handlelength=1.5,
    handletextpad=0.1,
    columnspacing=0.75,
    frameon=False,
)

fig.savefig("../figures/fig_result_jackknife.pdf", bbox_inches="tight")

In [ ]:
# Digitization of previous results
menard = np.array(
    [
        0.015372588004246894,
        0.16363373083250488,
        0.027147818672390675,
        0.08523850777558134,
        0.04794274447902959,
        0.03541057247645705,
        0.08584894051000302,
        0.03183392387526768,
        0.15372588004246884,
        0.013401938789527565,
        0.2714781867239069,
        0.008753798222678313,
        0.48612391624802315,
        0.007074748123832496,
        0.8584894051000298,
        0.004499654732304847,
        1.537258800424686,
        0.0033130745918951094,
        2.7147818672390636,
        0.0022823461023140207,
        4.861239162480231,
        0.0009867784425089342,
        0.015372588004246894,
        0.19454377598866124,
        0.027147818672390675,
        0.10269769440034786,
        0.04794274447902959,
        0.04440162288571723,
        0.08584894051000302,
        0.03734688077726687,
        0.15372588004246884,
        0.017029969604826688,
        0.2714781867239069,
        0.011123533691669145,
        0.48612391624802315,
        0.008638059376211973,
        0.8584894051000298,
        0.005794364085237468,
        1.537258800424686,
        0.00420995502849078,
        2.7147818672390636,
        0.0030183443573722673,
        4.794274447902952,
        0.0015514988038209934,
    ]
)
menard = menard.reshape(22, 2)

menard_x = menard[:11, 0]
menard_y = menard[:11, 1]

menard_err = menard[11:, 1] - menard_y

# TODO: Add upper limits
kids = np.array(
    [
        0.2952434728405713,
        0.011686711085103491,
        1.3939400795413572,
        0.002833398713044844,
        3.028837722476841,
        0.0017255322353215614,
        6.581242683055025,
        0.0009789696561929445,
        14.300124081209015,
        0.000716777914648752,
        0.2952434728405713,
        0.021191116846058046,
        1.3939400795413572,
        0.004458931178088874,
        3.028837722476841,
        0.0029148444562188815,
        6.581242683055025,
        0.0018261584682702627,
        14.300124081209015,
        0.0012456172541058495,
    ]
)
kids = kids.reshape(10, 2)

kids_x = kids[:5, 0]
kids_y = kids[:5, 1]

kids_err = kids[5:, 1] - kids_y

# Convert arcmin -> Mpc
arcmin_per_Mpc = 1e3 / (5.078 * 60)
kids_x /= arcmin_per_Mpc

# Convert Av -> reddening
Rv = 3.1
kids_y *= Rv
kids_err *= Rv

fig, ax1 = plt.subplots(figsize=(3, 3), dpi=200)

x = np.logspace(-2, 1, 100)
y = x**-0.86 * np.mean(menard_y / (menard_x**-0.86))
ax1.plot(x, y, c="k", ls="--", lw=0.75, zorder=-1)  # , label="Menard power law")

ax1.set(
    xscale="log",
    yscale="log",
    xlabel="$r_\perp$ (Mpc)",
    ylabel="$E(g-i)$ [mag]",
    ylim=(4e-4, 0.3),
    # xlim=(x.min(), x.max()),
)
ax1.set_box_aspect(1)

ax1.errorbar(
    menard_x,
    menard_y,
    yerr=menard_err,
    c="silver",
    mfc="w",
    ls="",
    marker=".",
    label="Menard 2010",
)

ax1.errorbar(
    kids_x,
    kids_y,
    yerr=kids_err,
    c="dimgray",
    mfc="w",
    ls="",
    marker=".",
    label="Genc 2025",
)

ax1.errorbar(
    x_mean,
    y_mean,
    yerr=y_err,
    color="r",
    label="Rubin DP1",
    ls="",
    marker="s",
    markersize=3,
)

ax1.legend(loc="upper right", fontsize=7, frameon=False)


def power_law(x, A, alpha):
    return A * x**alpha


popt, pcov = curve_fit(
    power_law, x_mean, y_mean, p0=[0.003, -0.86], sigma=y_cov, absolute_sigma=True
)
print("Best fit power law:")
print(f"    scale = {popt[0]:.1e} ± {np.sqrt(pcov[0, 0]):.1e}")
print(f"    index = {popt[1]:.2f} ± {np.sqrt(pcov[1, 1]):.2f}")
x_ = np.logspace(-2, 1, 100)
ax1.plot(x_, power_law(x_, *popt), c="r", ls="--", lw=0.75, zorder=-1)


# grid = np.geomspace(2e-2, 20, 10)
# for gi in grid:
#    ax1.axvline(gi, c="k", ls=":", lw=1, zorder=-2)

fig.savefig("../figures/fig_result_compare.pdf", bbox_inches="tight")

In [ ]:
results = load_jackknives("../results_main", "fcolors")

fig, ax = plt.subplots(figsize=(3, 3), dpi=200)

for i, color in enumerate(["g-i", "g-r", "r-i", "i-z"]):
    x = results[f"{color}_bin_centers"][0]

    y_all = -2.5 * np.log10(results[f"{color}_avg"])
    y = y_all.mean(axis=0)
    y_cov = np.cov(y_all.T, bias=True) * (len(y_all) - 1)
    y_err = np.sqrt(np.diag(y_cov))

    offset = 1 + ((i % 2) - 0.5) * 0.08
    ax.errorbar(
        x * offset,
        y,
        yerr=y_err,
        marker=".",
        ls="",
        label=f"${color}$",
        color=get_color(color),
    )

ax.set(xscale="log", yscale="log", ylim=(4e-4, 0.3))
ax.set(xlabel="$r_\perp$ [Mpc]", ylabel="$E(X-Y\,)$ [mag]")

# Create custom two-column legend
handles, labels = ax.get_legend_handles_labels()
blank = mpatches.Patch(color="none", label="")
new_handles = [handles[0], blank, blank, handles[1], handles[2], handles[3]]
new_labels = [labels[0], "", "", labels[1], labels[2], labels[3]]
ax.legend(
    new_handles,
    new_labels,
    ncol=2,
    frameon=False,
    handletextpad=0.1,
    columnspacing=0.5,
)

fig.savefig("../figures/fig_result_colors.pdf", bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3), dpi=200)

for i, subset in enumerate(["red", "blue"]):
    results = load_jackknives(f"../results_{subset}", "fcolors")

    x = results["g-i_bin_centers"][0]
    y_all = -2.5 * np.log10(results["g-i_avg"])
    y = y_all.mean(axis=0)
    y_cov = np.cov(y_all.T, bias=True) * (len(y_all) - 1)
    y_err = np.sqrt(np.diag(y_cov))

    # Plot upper limit for missing point
    if subset == "blue":
        ax.errorbar(
            x[1],
            y[1] + 3 * y_err[1],
            yerr=y_err[1],
            uplims=True,
            marker="_",
            color="b",
        )
        y[1] = np.nan

    offset = 1 + ((i % 2) - 0.5) * 0.08
    ax.errorbar(
        x * offset,
        y,
        yerr=y_err,
        marker=".",
        ls="",
        label=subset,
        color=subset,
    )

ax.set(xscale="log", yscale="log", ylim=(4e-4, 0.3))
ax.set(xlabel="$r_\perp$ [Mpc]", ylabel="$E(g-i)$ [mag]")
ax.legend(frameon=False, handletextpad=0.1)

# x_ = np.logspace(-2, 0, 100)
# ax.plot(x_, power_law(x_, *popt), c="k", ls="--", lw=0.75, zorder=-1)

fig.savefig("../figures/fig_result_red_vs_blue.pdf", bbox_inches="tight")